# 17 — Recomendação de ação para revisão humana

Este módulo não executa ações sobre clientes. Ele converte os indicadores em uma label de próximo passo e marca `revisao_humana=True` em todos os casos.

## Confiança comparável somente dentro de cada mecanismo

Como scores de modelo e heurística têm naturezas diferentes, o limiar serve apenas para direcionar casos frágeis à revisão manual; ele não é apresentado como probabilidade calibrada.

In [ ]:
def _has_low_confidence(
    sentiment: dict[str, Any],
    churn: dict[str, Any],
    opportunity: dict[str, Any],
) -> bool:
    relevant_scores = []
    if sentiment["label"] not in {"neutro", "misto"}:
        relevant_scores.append(float(sentiment["score"]))
    if churn["label"] != "baixo":
        relevant_scores.append(float(churn["score"]))
    if opportunity["label"] == "detectada":
        relevant_scores.append(float(opportunity["score"]))
    return any(score < 0.55 for score in relevant_scores)


## Ordem de prioridade

Churn alto aciona retenção. Depois vêm revisão manual, demonstração com produto explicitamente fundamentado, qualificação sem produto confiável e, por fim, acompanhamento da conta.

In [ ]:
def _recommend_action(
    products: list[dict[str, Any]],
    sentiment: dict[str, Any],
    churn: dict[str, Any],
    opportunity: dict[str, Any],
) -> dict[str, Any]:
    if churn["label"] == "alto":
        label = "acionar_retencao"
    elif sentiment["label"] == "misto" or _has_low_confidence(
        sentiment, churn, opportunity
    ):
        label = "revisar_manualmente"
    elif opportunity["label"] == "detectada":
        product_is_grounded = bool(
            products
            and products[0]["score"] >= 0.65
            and products[0]["explicit_match"]
            and products[0]["sources"]
        )
        label = (
            "agendar_demonstracao"
            if product_is_grounded
            else "qualificar_oportunidade"
        )
    else:
        label = "acompanhar_conta"
    return {"label": label, "revisao_humana": True}
